In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Dense, Flatten, Input, Reshape,
                                      Conv2D, Conv2DTranspose,
                                      BatchNormalization, LeakyReLU)
from tensorflow.keras.optimizers import Adam

# ============================================
# 1. DANE
# ============================================
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Dodanie wymiaru kanału dla CNN
x_train_cnn = x_train.reshape(-1, 28, 28, 1)
x_test_cnn = x_test.reshape(-1, 28, 28, 1)

# Flat dla klasyfikatora
x_train_flat = x_train.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

# ============================================
# 2. KLASYFIKATOR
# ============================================
classifier = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

classifier.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

classifier.fit(x_train, y_train, epochs=10, batch_size=128,
               validation_split=0.1, verbose=1)

loss_orig, acc_orig = classifier.evaluate(x_test, y_test, verbose=0)
print(f"Dokładność na ORYGINALNYCH: {acc_orig*100:.2f}%")

# ============================================
# 3. CONVOLUTIONAL AUTOENCODER
# ============================================
latent_dim = 2

# ENCODER (CNN)
encoder_input = Input(shape=(28, 28, 1))
x = Conv2D(32, 3, strides=2, padding='same')(encoder_input)
x = LeakyReLU(negative_slope=0.2)(x)
x = BatchNormalization()(x)
x = Conv2D(64, 3, strides=2, padding='same')(x)
x = LeakyReLU(negative_slope=0.2)(x)
x = BatchNormalization()(x)
x = Conv2D(128, 3, strides=1, padding='same')(x)
x = LeakyReLU(negative_slope=0.2)(x)
x = BatchNormalization()(x)
x = Flatten()(x)
x = Dense(256)(x)
x = LeakyReLU(negative_slope=0.2)(x)
latent = Dense(latent_dim, activation='tanh', name='latent')(x)

encoder = Model(encoder_input, latent, name='encoder')

# DECODER (Transposed CNN)
decoder_input = Input(shape=(latent_dim,))
x = Dense(256)(decoder_input)
x = LeakyReLU(negative_slope=0.2)(x)
x = Dense(7 * 7 * 128)(x)
x = LeakyReLU(negative_slope=0.2)(x)
x = Reshape((7, 7, 128))(x)
x = Conv2DTranspose(64, 3, strides=2, padding='same')(x)
x = LeakyReLU(negative_slope=0.2)(x)
x = BatchNormalization()(x)
x = Conv2DTranspose(32, 3, strides=2, padding='same')(x)
x = LeakyReLU(negative_slope=0.2)(x)
x = BatchNormalization()(x)
decoder_output = Conv2DTranspose(1, 3, strides=1, padding='same', activation='sigmoid')(x)

decoder = Model(decoder_input, decoder_output, name='decoder')

# AUTOENCODER
autoencoder_input = Input(shape=(28, 28, 1))
encoded = encoder(autoencoder_input)
decoded = decoder(encoded)
autoencoder = Model(autoencoder_input, decoded, name='autoencoder')

autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy')

print("\n--- Trenowanie CNN AutoEncodera ---")
autoencoder.fit(x_train_cnn, x_train_cnn,
                epochs=50,
                batch_size=128,
                validation_split=0.1,
                verbose=1)

# ============================================
# 4. REKONSTRUKCJA I PORÓWNANIE
# ============================================
x_test_reconstructed = autoencoder.predict(x_test_cnn).reshape(-1, 28, 28)

loss_recon, acc_recon = classifier.evaluate(x_test_reconstructed, y_test, verbose=0)
print(f"\nDokładność na ORYGINALNYCH: {acc_orig*100:.2f}%")
print(f"Dokładność na ZREKONSTRUOWANYCH: {acc_recon*100:.2f}%")

# ============================================
# 5. WIZUALIZACJE
# ============================================

# Rekonstrukcje
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(x_test[i], cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(x_test_reconstructed[i], cmap='gray')
    axes[1, i].axis('off')
plt.savefig('reconstruction_comparison.png', dpi=150)
plt.show()

# Przestrzeń latentna
z_test = encoder.predict(x_test_cnn)
plt.figure(figsize=(10, 8))
scatter = plt.scatter(z_test[:, 0], z_test[:, 1], c=y_test, cmap='tab10', s=1, alpha=0.5)
plt.colorbar(scatter, label='Cyfra')
plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)
plt.savefig('latent_space_distribution.png', dpi=150)
plt.show()

# Wizualizacja dekodera
n = 20
figure = np.zeros((28 * n, 28 * n))
grid_x = np.linspace(-0.95, 0.95, n)
grid_y = np.linspace(-0.95, 0.95, n)[::-1]

for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = np.array([[xi, yi]])
        x_decoded = decoder.predict(z_sample, verbose=0)
        figure[i*28:(i+1)*28, j*28:(j+1)*28] = x_decoded[0].reshape(28, 28)

plt.figure(figsize=(12, 12))
plt.imshow(figure, cmap='gray')
plt.savefig('latent_space_visualization.png', dpi=150)
plt.show()


Rozmiar zbioru treningowego: (60000, 28, 28)
Rozmiar zbioru testowego: (10000, 28, 28)

--- Trenowanie klasyfikatora ---
Epoch 1/10
422/422 [==============================] - 18s 25ms/step - loss: 0.2951 - accuracy: 0.9132 - val_loss: 0.1150 - val_accuracy: 0.9683
Epoch 2/10
334/422 [======================>.......] - ETA: 1s - loss: 0.1080 - accuracy: 0.9676


KeyboardInterrupt

